[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/05_attention.ipynb)

# 🔴 Hard: Softmax Attention

Реализуйте базовый механизм внимания, используемый в Transformer. Здесь `Q` можно понимать как запросы «что я ищу?», `K` — как адреса «что у меня есть?», а `V` — как содержимое, которое будет смешано в ответе.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

## Как работает формула

1. `Q @ K.transpose(-2, -1)` сравнивает каждый запрос с каждым ключом и даёт оценки формы `(B, S_q, S_k)`.
2. Деление на $\sqrt{d_k}$ удерживает разброс оценок под контролем. Без масштаба большие значения $d_k$ чаще загоняют softmax в насыщение, где градиенты малы.
3. Softmax применяется по последней оси — по всем ключам для одного запроса. Поэтому каждая строка весов неотрицательна и суммируется в 1.
4. Умножение весов на `V` создаёт взвешенную сумму значений формы `(B, S_q, D_v)`. Для каждого запроса результат лежит в выпуклой оболочке соответствующих векторов `V`.

### Пример с формами

Пусть `Q.shape == (2, 3, 64)`, `K.shape == (2, 5, 64)`, а `V.shape == (2, 5, 32)`. Тогда матрица оценок и весов имеет форму `(2, 3, 5)`, а результат — `(2, 3, 32)`. Это cross-attention: три позиции-запроса читают информацию из пяти позиций-источников. Если одна строка оценок после softmax равна `[0.1, 0.7, 0.2]`, выход этой строки равен `0.1*V0 + 0.7*V1 + 0.2*V2`.

### Signature
```python
def scaled_dot_product_attention(
    Q: torch.Tensor,  # (batch, seq_q, d_k)
    K: torch.Tensor,  # (batch, seq_k, d_k)
    V: torch.Tensor,  # (batch, seq_k, d_v)
) -> torch.Tensor:   # (batch, seq_q, d_v)
    ...
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- You **may** use `torch.softmax` and `torch.bmm`
- Must support autograd
- Must handle cross-attention (seq_q ≠ seq_k)

## План реализации

- Проверьте, какие две последние оси должны участвовать в матричном произведении.
- Получите scores и масштабируйте их Python-числом или скаляром на том же устройстве.
- Нормализуйте scores строго по оси `S_k`.
- Умножьте веса внимания на `V`; не создавайте циклы по batch или позициям.

## Быстрые проверки

- При `S_q != S_k` форма результата должна быть `(B, S_q, D_v)`.
- Если все ключи одинаковы, веса одной строки должны быть равномерными.
- Если `S_k == 1`, результат для каждого запроса совпадает с единственным `V`.
- Вызов `output.sum().backward()` должен заполнить градиенты `Q`, `K` и `V`.

## Частые ошибки

- Softmax по оси запросов вместо оси ключей.
- Транспонирование batch-оси вместе с `K`.
- Масштабирование на $\sqrt{D_v}$ вместо $\sqrt{D_k}$.
- Предположение, что `S_q` обязательно равно `S_k`.

## Материалы для глубокого изучения

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) — исходная статья о Transformer, раздел 3.2.1.
- [PyTorch: scaled_dot_product_attention](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html) — эталонный контракт, маски и современные backend-реализации.
- [PyTorch SDPA tutorial](https://docs.pytorch.org/tutorials/intermediate/scaled_dot_product_attention_tutorial.html) — связь формулы с эффективными ядрами.

## Вопросы для самопроверки

1. Почему softmax нормализуется по `S_k`, а не по `S_q`?
2. Как изменится форма ответа, если `D_v != D_k`?
3. Почему масштабирование особенно важно при большом `D_k`?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import math

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def scaled_dot_product_attention(Q, K, V):
    pass  # Replace this

In [ ]:
# 🧪 Debug
torch.manual_seed(42)
Q = torch.randn(2, 4, 8)
K = torch.randn(2, 4, 8)
V = torch.randn(2, 4, 8)

out = scaled_dot_product_attention(Q, K, V)
print("Output shape:", out.shape)          # should be (2, 4, 8)
print("Has NaN?    ", torch.isnan(out).any().item())  # should be False
print("Has Inf?    ", torch.isinf(out).any().item())  # should be False

# Cross-attention: seq_q != seq_k
Q2 = torch.randn(1, 3, 16)
K2 = torch.randn(1, 5, 16)
V2 = torch.randn(1, 5, 32)
out2 = scaled_dot_product_attention(Q2, K2, V2)
print("Cross-attn shape:", out2.shape)     # should be (1, 3, 32)

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check("attention")